In [34]:
# various import statements
import os
from dotenv import load_dotenv
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import matplotlib.pyplot as plt 
import seaborn as sbn
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

In [35]:
# load in data from google sheet where my scores are stored
load_dotenv()
scope = ['https://www.googleapis.com/auth/drive']
jsonpath = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
spreadsheet_id = '1suD1XqF5Dr5PdR3ggxxIyOo2_6vYakOnH1Afha72SR0'
creds = Credentials.from_service_account_file(jsonpath, scopes=scope)
gc = gspread.authorize(creds)
spreadsheet = gc.open_by_key(spreadsheet_id)
worksheet = spreadsheet.sheet1

In [36]:
data = worksheet.get_all_values()
score_df = pd.DataFrame(data[1:], columns=data[0])
score_df.head()

,Restaurant Name,City,Neighborhood,Cuisine,Sub Cuisine,Visit Date,Ingredient Quality,Execution,Personality,Price to Quality,Price to Portion,Sensory,Design,Vibe,Pacing,Staff
0,Akahoshi Ramen,Chicago,Logan Square,Japanese,Ramen,3/20/2026,8,9,8,8,7,7,5,7,7,7
1,Cabra,Chicago,Fulton Market,Peruvian,,3/21/2026,6,7,7,6,6,4,6,7,5,6
2,Momotaro,Chicago,Fulton Market,Japanese,,3/27/2026,8,8,7,7,4,6,9,8,7,8


In [37]:
# score weighting definition

# food score weighting
ingredient_qual_weight = 0.3
execution_weight = 0.4
personality_weight = 0.3

# value score weighting
price_to_quality = 0.7
price_to_portion = 0.3

# atmosphere weighting
sensory_weight = 0.3
design_weight = 0.3
vibe_weight = 0.4

# hospitality weighting
pacing_weight = 0.3
staff_weight = 0.7

# overall score weighting
food_weight = 0.4
value_weight = 0.2
atmosphere_weight = 0.2
hospitality_weight = 0.2

# define variable to grab just score columns
cat_score_cols = [
    'Ingredient Quality', 
    'Execution', 
    'Personality', 
    'Price to Quality', 
    'Price to Portion', 
    'Sensory', 
    'Design', 
    'Vibe', 
    'Pacing', 
    'Staff'
]

# ensure each score col is int
score_df[cat_score_cols] = score_df[cat_score_cols].astype(int)

In [38]:
# score calculation

score_df['Food Score'] = (
    score_df['Ingredient Quality'] * ingredient_qual_weight +
    score_df['Execution'] * execution_weight +
    score_df['Personality'] * personality_weight
).round(1)

score_df['Value Score'] = (
    score_df['Price to Quality'] * price_to_quality +
    score_df['Price to Portion'] * price_to_portion
).round(1)

score_df['Atmosphere Score'] = (
    score_df['Sensory'] * sensory_weight +
    score_df['Design'] * design_weight +
    score_df['Vibe'] * vibe_weight
).round(1)

score_df['Hospitality Score'] = (
    score_df['Pacing'] * pacing_weight +
    score_df['Staff'] * staff_weight
).round(1)

# overall score calculation
score_df['Overall Score'] = (
    score_df['Food Score'] * food_weight +
    score_df['Value Score'] * value_weight +
    score_df['Atmosphere Score'] * atmosphere_weight +
    score_df['Hospitality Score'] * hospitality_weight
).round(1)

score_df.head()

,Restaurant Name,City,Neighborhood,Cuisine,Sub Cuisine,Visit Date,Ingredient Quality,Execution,Personality,Price to Quality,...,Sensory,Design,Vibe,Pacing,Staff,Food Score,Value Score,Atmosphere Score,Hospitality Score,Overall Score
0,Akahoshi Ramen,Chicago,Logan Square,Japanese,Ramen,3/20/2026,8,9,8,8,...,7,5,7,7,7,8.4,7.7,6.4,7.0,7.6
1,Cabra,Chicago,Fulton Market,Peruvian,,3/21/2026,6,7,7,6,...,4,6,7,5,6,6.7,6.0,5.8,5.7,6.2
2,Momotaro,Chicago,Fulton Market,Japanese,,3/27/2026,8,8,7,7,...,6,9,8,7,8,7.7,6.1,7.7,7.7,7.4


In [50]:
# code to generate the spider chart used in articles

# one spider chart vs cuisine average, one spider chart vs city average

# update this line to generate associated charts
curr_review = 'Momotaro'

# get score values for current selection only
curr_review_scores = score_df.loc[score_df['Restaurant Name'] == curr_review, cat_score_cols].iloc[0]

# create list of just current review values for spider chart
review_vals = curr_review_scores.values.tolist()
review_vals.append(review_vals[0])

# get average score values for current selection cuisine
curr_cuisine = score_df.loc[score_df['Restaurant Name'] == curr_review, 'Cuisine'].values[0]
cuisine_avgs = score_df.groupby('Cuisine')[cat_score_cols].mean()
curr_cuisine_avg = cuisine_avgs.loc[curr_cuisine]

# create list for shared categories
categories = curr_cuisine_avg.index.tolist()
categories.append(categories[0])

# create list of scores for 
avg_vals = curr_cuisine_avg.values.tolist()
avg_vals.append(avg_vals[0])

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=avg_vals,
    theta=categories,
    fill='toself',
    name=f'{curr_cuisine} Average',
    fillcolor='rgba(150, 150, 150, 0.3)',
    line=dict(color='gray')
))

fig.add_trace(go.Scatterpolar(
    r=review_vals,
    theta=categories,
    fill='toself',
    name=curr_review,
    fillcolor='rgba(255, 0, 0, 0.25)',
    line=dict(color='red', width=2)
))

styled_title = f"<span style='color: red; font-weight: bold;'>{curr_review}</span> vs. {curr_cuisine} Cuisine Averages"

fig.update_layout(
    title=styled_title,
    title_x=0.5,
    width=600,
    height=500,
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 10]
        )
    ),
    showlegend=False
)
fig.show()
